# VigorAI — Huấn luyện AI Model B (`YOLOv8s` 16/18 Món `F19`)

> **Owner:** `M4` · **Task:** `T-M4-06`  
> Chọn **Runtime -> Change runtime type -> T4 GPU** trước khi chạy.  
> Hỗ trợ tự động **cả 2 cách**: upload trực tiếp `model_b_v3_colab.zip` vào `/content/` **hoặc** đọc từ Google Drive `MyDrive/model_b_v3_colab.zip` / `MyDrive/model_b_v3.zip`.

In [ ]:
# 1. Kiem tra GPU & cai dat thu vien
!nvidia-smi
!pip install -q ultralytics onnx onnxslim onnxruntime

In [ ]:
# 2. Tim va giai nen dataset (tu /content/ hoac Google Drive)
import os, zipfile, glob

candidates = [
    '/content/model_b_v3_colab.zip',
    '/content/model_b_v3.zip',
]
zip_path = next((p for p in candidates if os.path.isfile(p)), None)

if zip_path is None:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_candidates = [
        '/content/drive/MyDrive/model_b_v3_colab.zip',
        '/content/drive/MyDrive/model_b_v3.zip',
    ]
    zip_path = next((p for p in drive_candidates if os.path.isfile(p)), None)

assert zip_path is not None, 'Hay upload model_b_v3_colab.zip vao /content/ hoac MyDrive!'
print('Dang giai nen:', zip_path)
!rm -rf /content/data && mkdir -p /content/data
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('/content/data')
!ls -la /content/data

In [ ]:
# 3. Xac dinh train_yolo.py & data.yaml (uu tien ban dong goi san trong zip)
import glob, os, subprocess

yamls = glob.glob('/content/data/**/data.yaml', recursive=True)
assert yamls, 'Khong tim thay data.yaml trong /content/data'
DATA_YAML = yamls[0]

scripts = glob.glob('/content/data/**/train_yolo.py', recursive=True)
if scripts:
    TRAIN_SCRIPT = scripts[0]
else:
    !rm -rf /content/VigorAI && git clone -b vu --depth 1 https://github.com/HPhucTV/VigorAI.git /content/VigorAI
    TRAIN_SCRIPT = '/content/VigorAI/ai_research/model_b_food_yolo/train_yolo.py'

print('DATA_YAML   :', DATA_YAML)
print('TRAIN_SCRIPT:', TRAIN_SCRIPT)

In [ ]:
# 4. Huan luyen YOLOv8s (80 epochs, patience=20) + test + xuat ONNX
!rm -rf /content/runs
!python "{TRAIN_SCRIPT}" \
    --data "{DATA_YAML}" \
    --model yolov8s.pt \
    --epochs 80 \
    --imgsz 640 \
    --batch 16 \
    --patience 20 \
    --project /content/runs \
    --name model_b_v3 \
    --version yolov8s-gymfood-v1.1 \
    --report /content/runs/model_b_v3_train_report.json

In [ ]:
# 5. Gom ket qua ve /content/output_v1_1 (va luu Drive neu da mount)
!rm -rf /content/output_v1_1 && mkdir -p /content/output_v1_1
!cp /content/runs/model_b_v3/weights/best.onnx /content/output_v1_1/best.onnx
!cp /content/runs/model_b_v3/weights/best.pt /content/output_v1_1/best.pt
!cp /content/runs/model_b_v3_train_report.json /content/output_v1_1/model_b_v3_train_report.json
!cp /content/runs/model_b_manifest_generated.json /content/output_v1_1/model_b_manifest_generated.json
!cp /content/runs/model_b_v3/results.png /content/output_v1_1/results.png
!cp /content/runs/model_b_v3_test/confusion_matrix*.png /content/output_v1_1/ 2>/dev/null || true

import os
if os.path.isdir('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/VigorAI_ModelB_Output
    !cp -r /content/output_v1_1/* /content/drive/MyDrive/VigorAI_ModelB_Output/
!ls -lh /content/output_v1_1